In [9]:
import zipfile
import io
import os
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

## 1. Загрузка фида

Ищем GTFS-zip в `data/`. Если не найден — скачиваем с transport.data.gouv.fr (официальный реестр открытых транспортных данных Франции).

In [10]:
import requests
from pathlib import Path

GTFS_PATH = Path("data/tam_gtfs.zip")
TARGET_SLUG = "offre-de-transport-de-montpellier-mediterranee-metropole-tam-gtfs"

# Получаем метаданные через list-эндпоинт (single-dataset endpoint не поддерживает slug)
print("Запрашиваем API transport.data.gouv.fr...")
all_datasets = requests.get(
    "https://transport.data.gouv.fr/api/datasets?q=Montpellier",
    timeout=20,
).json()

dataset = next((d for d in all_datasets if d.get("slug") == TARGET_SLUG), None)
if dataset is None:
    raise RuntimeError(f"Датасет {TARGET_SLUG!r} не найден через API")

gtfs_resources = [r for r in dataset.get("resources", []) if r.get("format") == "GTFS"]
if not gtfs_resources:
    raise RuntimeError("GTFS-ресурс не найден в датасете")

resource = gtfs_resources[0]
api_meta = resource.get("metadata", {})

print(f"Датасет:   {dataset['title']}")
print(f"Ресурс:    {resource['title']}")
print(f"Обновлён:  {resource['updated'][:10]}")
print(f"Период:    {api_meta.get('start_date')} → {api_meta.get('end_date')}")
print(f"has_shapes:{api_meta.get('has_shapes')}")

if not GTFS_PATH.exists():
    download_url = resource.get("original_url") or resource["url"]
    print(f"\nСкачиваем: {download_url}")
    r = requests.get(download_url, timeout=120, allow_redirects=True)
    r.raise_for_status()
    GTFS_PATH.write_bytes(r.content)
    print(f"Сохранено: {GTFS_PATH} ({len(r.content) / 1024:.0f} KB)")
else:
    print(f"\nФайл уже есть: {GTFS_PATH} ({GTFS_PATH.stat().st_size / 1024:.0f} KB)")

Запрашиваем API transport.data.gouv.fr...
Датасет:   Réseau urbain et suburbain TaM
Ресурс:    Offre de transport TAM en GTFS
Обновлён:  2026-07-03
Период:    2026-07-13 → 2026-10-16
has_shapes:False

Скачиваем: https://data.montpellier3m.fr/sites/default/files/ressources/TAM_MMM_GTFS.zip


ChunkedEncodingError: ('Connection broken: IncompleteRead(0 bytes read, 3708109 more expected)', IncompleteRead(0 bytes read, 3708109 more expected))

## 2. Состав архива: есть ли обязательные файлы и shapes.txt

In [ ]:
REQUIRED = {
    "agency.txt", "stops.txt", "routes.txt",
    "trips.txt", "stop_times.txt", "calendar.txt",
}
OPTIONAL_KEY = {"feed_info.txt", "shapes.txt", "calendar_dates.txt", "transfers.txt"}

with zipfile.ZipFile(GTFS_PATH) as z:
    files = {Path(n).name: z.getinfo(n).file_size for n in z.namelist() if not n.endswith("/")}

rows = []
for fname, size in sorted(files.items()):
    status = "обязательный" if fname in REQUIRED else ("ключевой" if fname in OPTIONAL_KEY else "дополнительный")
    rows.append({"файл": fname, "размер (KB)": round(size / 1024, 1), "статус": status})

contents = pd.DataFrame(rows)

has_shapes = "shapes.txt" in files
missing = REQUIRED - set(files)

print(f"Файлов в архиве: {len(files)}")
print(f"shapes.txt: {'✓ есть' if has_shapes else '✗ нет'}")
print(f"Отсутствующие обязательные: {missing if missing else 'нет'}")
contents

## 3. Дата фида (feed_info.txt) и период действия (calendar.txt)

In [ ]:
with zipfile.ZipFile(GTFS_PATH) as z:
    # feed_info.txt
    if "feed_info.txt" in files:
        feed_info = pd.read_csv(z.open("feed_info.txt"), dtype=str)
        print("=== feed_info.txt ===")
        display(feed_info)
    else:
        print("feed_info.txt отсутствует")
        feed_info = None

    # calendar.txt — диапазон действия расписания
    if "calendar.txt" in files:
        cal = pd.read_csv(z.open("calendar.txt"), dtype=str)
        cal["start_date"] = pd.to_datetime(cal["start_date"], format="%Y%m%d")
        cal["end_date"]   = pd.to_datetime(cal["end_date"],   format="%Y%m%d")
        print(f"\n=== calendar.txt ===")
        print(f"Период расписания: {cal['start_date'].min().date()} → {cal['end_date'].max().date()}")
        print(f"Записей (service_id): {len(cal)}")
    
    # calendar_dates.txt — исключения
    if "calendar_dates.txt" in files:
        cal_dates = pd.read_csv(z.open("calendar_dates.txt"), dtype=str)
        print(f"\ncalendar_dates.txt: {len(cal_dates)} исключений")

## 4. Базовая статистика фида

In [ ]:
with zipfile.ZipFile(GTFS_PATH) as z:
    agency     = pd.read_csv(z.open("agency.txt"),     dtype=str)
    routes     = pd.read_csv(z.open("routes.txt"),     dtype=str)
    trips      = pd.read_csv(z.open("trips.txt"),      dtype=str)
    stops      = pd.read_csv(z.open("stops.txt"),      dtype=str)
    stop_times = pd.read_csv(z.open("stop_times.txt"), dtype=str)
    shapes     = pd.read_csv(z.open("shapes.txt"),     dtype=str) if has_shapes else None

route_type_names = {
    "0": "трамвай", "1": "метро", "2": "ж/д",
    "3": "автобус", "7": "фуникулёр", "11": "троллейбус",
}

stats = {
    "Оператор":             agency["agency_name"].iloc[0] if len(agency) else "—",
    "Маршрутов":            len(routes),
    "Рейсов (trips)":       len(trips),
    "Остановок":            len(stops),
    "Записей stop_times":   len(stop_times),
    "shapes.txt":           f"есть ({shapes['shape_id'].nunique()} shape_id)" if shapes is not None else "нет",
}

# Типы маршрутов
if "route_type" in routes.columns:
    for rtype, count in routes["route_type"].value_counts().items():
        label = route_type_names.get(str(rtype), f"тип {rtype}")
        stats[f"  {label}"] = count

pd.DataFrame.from_dict(stats, orient="index", columns=["значение"])

## 5. Итоговая карточка фида

In [ ]:
feed_start = feed_info["feed_start_date"].iloc[0] if feed_info is not None and "feed_start_date" in feed_info.columns else "н/д"
feed_end   = feed_info["feed_end_date"].iloc[0]   if feed_info is not None and "feed_end_date"   in feed_info.columns else "н/д"
feed_ver   = feed_info["feed_version"].iloc[0]    if feed_info is not None and "feed_version"    in feed_info.columns else "н/д"

print("=" * 45)
print("GTFS TaM Montpellier — карточка фида")
print("=" * 45)
print(f"Статус:       {'есть ✓' if GTFS_PATH.exists() else 'нет ✗'}")
print(f"Источник:     {GTFS_PATH}")
print(f"Версия фида:  {feed_ver}")
print(f"Период:       {feed_start} → {feed_end}")
print(f"shapes.txt:   {'есть ✓' if has_shapes else 'нет ✗'}")
print(f"Маршрутов:    {len(routes)}")
print(f"Остановок:    {len(stops)}")
print(f"Рейсов:       {len(trips)}")
print("=" * 45)